In [0]:
# ════════════════════════════════════════════════════════════════
# run_framework_v2.py - OPTIMIZED FOR NEW SCHEMA
# ════════════════════════════════════════════════════════════════
# Supports:
#   - Unified schema: data_flow_l0_detail, data_flow_pb_detail
#   - All layers: L0 (ingestion), L1 (bronze), L2 (silver)
#   - Load types: FULL, INCREMENTAL, MERGE
#   - File formats: csv, json, parquet, delta, excel
# ════════════════════════════════════════════════════════════════

In [0]:
# PARAMETERS
dbutils.widgets.text("GROUP_ID", "")
dbutils.widgets.text("TARGET_LOAD_TABLE", "")
dbutils.widgets.text("ENVIRONMENT", "dev")

GROUP_ID = dbutils.widgets.get("GROUP_ID").strip().upper()
TARGET_TABLE = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENV = dbutils.widgets.get("ENVIRONMENT").strip()

# Determine layer from GROUP_ID suffix
if GROUP_ID.endswith("_L0"):
    LAYER = "L0"
elif GROUP_ID.endswith("_L1"):
    LAYER = "L1"
elif GROUP_ID.endswith("_L2"):
    LAYER = "L2"
else:
    LAYER = "ALL"

if not GROUP_ID:
    raise ValueError("GROUP_ID is required")

print(f"GROUP_ID      : {GROUP_ID}")
print(f"LAYER         : {LAYER}")
print(f"TARGET_TABLE  : {TARGET_TABLE or 'ALL'}")
print(f"ENVIRONMENT   : {ENV}")

In [0]:
# IMPORTS
import traceback
from datetime import datetime
from pyspark.sql import functions as F

# Hard-code catalog to avoid serverless permissions issues
CATALOG = "demo_catalog"

print(f"CATALOG       : {CATALOG}")

In [0]:
# READ SOURCE
def read_source(url, fmt="csv", delimiter=","):
    """
    Read from HTTP/S3/DBFS/Volumes into Spark DataFrame.
    Supports: csv, json, parquet, delta
    """
    fmt = (fmt or "csv").strip().lower()
    url = url.strip()
    
    if not url:
        raise ValueError("Source URL is empty")
    
    print(f"  Reading [{fmt}] from: {url[:80]}...")
    
    # HTTP/HTTPS sources
    if url.startswith("http"):
        import requests, io, pandas as pd
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        if fmt == "csv":
            pdf = pd.read_csv(io.BytesIO(response.content), sep=delimiter)
            return spark.createDataFrame(pdf)
        elif fmt == "json":
            pdf = pd.read_json(io.BytesIO(response.content))
            return spark.createDataFrame(pdf)
        elif fmt == "parquet":
            pdf = pd.read_parquet(io.BytesIO(response.content))
            return spark.createDataFrame(pdf)
        else:
            raise ValueError(f"Unsupported HTTP format: {fmt}")
    
    # Cloud storage paths
    else:
        if fmt == "csv":
            return spark.read.option("header", "true").option("inferSchema", "true").option("sep", delimiter).csv(url)
        elif fmt == "json":
            return spark.read.json(url)
        elif fmt == "parquet":
            return spark.read.parquet(url)
        elif fmt == "delta":
            return spark.read.format("delta").load(url)
        else:
            raise ValueError(f"Unsupported format: {fmt}")

In [0]:
# WRITE TABLE
def write_table(df, catalog, schema, table, load_type="FULL", merge_keys=None):
    """
    Write DataFrame to Delta table.
    Returns row count.
    """
    # Create schema if needed
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    
    full_name = f"{catalog}.{schema}.{table}"
    load_type = (load_type or "FULL").strip().upper()
    
    if load_type == "FULL":
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
        return df.count()
    
    elif load_type in ("INCREMENTAL", "APPEND"):
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_name)
        return df.count()
    
    elif load_type == "MERGE":
        if not merge_keys:
            raise ValueError(f"MERGE_KEYS required for MERGE load type")
        
        keys = [k.strip() for k in merge_keys.split(",")]
        temp_view = f"_tmp_{table}"
        df.createOrReplaceTempView(temp_view)
        
        # Create table if not exists
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_name} USING DELTA AS SELECT * FROM {temp_view} WHERE 1=0")
        
        # Build MERGE statement
        on_clause = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        update_cols = [c for c in df.columns if c not in keys]
        update_set = ", ".join([f"target.{c} = source.{c}" for c in update_cols])
        insert_cols = ", ".join(df.columns)
        insert_vals = ", ".join([f"source.{c}" for c in df.columns])
        
        merge_sql = f"""
            MERGE INTO {full_name} AS target
            USING {temp_view} AS source
            ON {on_clause}
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
        """
        spark.sql(merge_sql)
        return df.count()
    
    else:
        raise ValueError(f"Unsupported load type: {load_type}")

In [0]:
# AUDIT LOG
def write_audit(group_id, table_name, layer, status, message, rows, start_time, end_time):
    """
    Write audit record. Never raises.
    """
    try:
        safe_msg = str(message).replace("'", "''")[:400]
        start_ts = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_ts = end_time.strftime("%Y-%m-%d %H:%M:%S")
        
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log (
                DATA_FLOW_GROUP_ID,
                TARGET_TABLE,
                STATUS,
                MESSAGE,
                ETL_LAYER,
                ROWS_PROCESSED,
                START_TIME,
                END_TIME,
                LOAD_TS
            )
            VALUES (
                '{group_id}',
                '{table_name}',
                '{status}',
                '{safe_msg}',
                '{layer}',
                {rows},
                '{start_ts}',
                '{end_ts}',
                current_timestamp()
            )
        """)
    except Exception as e:
        print(f"  ⚠ Audit write failed: {e}")

In [0]:
# PROCESS LAYER - REWRITTEN WITH PROPER ERROR HANDLING
def process_layer(detail_table, layer):
    """
    Process all objects for a given layer.
    Raises exceptions on failure - NO SILENT ERRORS.
    
    Args:
        detail_table: Control table name (data_flow_l0_detail or data_flow_pb_detail)
        layer: L0, L1, or L2
    
    Returns:
        True if all objects processed successfully
    
    Raises:
        ValueError: If configuration is invalid
        Exception: If processing fails and cannot continue
    """
    print(f"\n{'='*60}")
    print(f"  LAYER: {layer} | TABLE: {detail_table}")
    print(f"{'='*60}")
    
    # Validate inputs
    if layer not in ["L0", "L1", "L2"]:
        raise ValueError(f"Invalid layer '{layer}'. Must be L0, L1, or L2.")
    
    if detail_table not in ["data_flow_l0_detail", "data_flow_pb_detail"]:
        raise ValueError(f"Invalid control table '{detail_table}'.")
    
    # Build query based on layer
    # CRITICAL: pb_detail table does NOT have ETL_LAYER column!
    # The layer is implicit based on which table you query.
    if layer == "L0":
        obj_col = "SOURCE_OBJ_NAME"
        table_filter = f"AND SOURCE_OBJ_NAME = '{TARGET_TABLE}'" if TARGET_TABLE and TARGET_TABLE.upper() != "ALL" else ""
    else:  # L1 or L2
        obj_col = "TARGET_OBJ_NAME"
        table_filter = f"AND TARGET_OBJ_NAME = '{TARGET_TABLE}'" if TARGET_TABLE and TARGET_TABLE.upper() != "ALL" else ""
    
    # Query control table (NO layer_filter - pb_detail doesn't have ETL_LAYER)
    query = f"""
        SELECT * 
        FROM {CATALOG}.admin.{detail_table}
        WHERE DATA_FLOW_GROUP_ID = '{GROUP_ID}'
          AND IS_ACTIVE = 'Y'
          {table_filter}
        ORDER BY COALESCE(PRIORITY, 999), {obj_col}
    """
    
    print(f"  Executing query:")
    print(f"    {query.strip().replace(chr(10), ' ')}")
    
    try:
        rows = spark.sql(query).collect()
    except Exception as e:
        error_msg = f"Failed to query control table {detail_table}: {type(e).__name__}: {str(e)}"
        print(f"  ❌ {error_msg}")
        raise RuntimeError(error_msg) from e
    
    if not rows:
        print(f"  ⚠ No active objects found for {layer}")
        return True  # No work to do is not a failure
    
    print(f"  Objects to process: {len(rows)}\n")
    
    # Track failures
    failed_objects = []
    success_count = 0
    
    for idx, row in enumerate(rows, 1):
        r = row.asDict()
        t0 = datetime.now()
        status = "FAILED"
        msg = ""
        count = 0
        target_table = None  # Initialize for finally block
        
        try:
            if layer == "L0":
                # ═══════════════════════════════════════════════════
                # L0: File Ingestion
                # ═══════════════════════════════════════════════════
                source_url = (r.get("SOURCE") or "").strip()
                target_schema = (r.get("SOURCE_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                file_format = (r.get("INPUT_FILE_FORMAT") or "csv").strip().lower()
                load_type = (r.get("LOAD_TYPE") or "FULL").strip().upper()
                delimiter = (r.get("DELIMETER") or ",").strip()
                
                # Validate required fields
                if not source_url:
                    raise ValueError("SOURCE URL is required but empty")
                if not target_schema:
                    raise ValueError("SOURCE_OBJ_SCHEMA is required but empty")
                if not target_table:
                    raise ValueError("SOURCE_OBJ_NAME is required but empty")
                
                # Strip file extension from table name
                import os
                target_table = os.path.splitext(target_table)[0]
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  [{idx}/{len(rows)}] ▶ {full_name}")
                print(f"    Source: {source_url}")
                print(f"    Format: {file_format} | Load: {load_type}")
                
                # Read source
                try:
                    df = read_source(source_url, file_format, delimiter)
                except Exception as e:
                    raise RuntimeError(f"Failed to read source: {type(e).__name__}: {str(e)}") from e
                
                # Validate DataFrame
                if df is None:
                    raise ValueError("read_source returned None")
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write
                try:
                    count = write_table(df, CATALOG, target_schema, target_table, load_type)
                except Exception as e:
                    raise RuntimeError(f"Failed to write table: {type(e).__name__}: {str(e)}") from e
                
                if count == 0:
                    print(f"    ⚠ WARNING: 0 rows written")
                
                status = "SUCCESS"
                msg = f"{count:,} rows loaded"
                print(f"    ✅ {msg}")
                success_count += 1
                
            else:
                # ═══════════════════════════════════════════════════
                # L1/L2: Transformation
                # ═══════════════════════════════════════════════════
                target_schema = (r.get("TARGET_OBJ_SCHEMA") or "").strip()
                target_table = (r.get("TARGET_OBJ_NAME") or "").strip()
                transform_query = (r.get("TRANSFORM_QUERY") or "").strip()
                load_type = (r.get("LOAD_TYPE") or "FULL").strip().upper()
                merge_keys = (r.get("TARGET_PK") or r.get("MERGE_KEYS") or "").strip()
                source_schema = (r.get("SOURCE_OBJ_SCHEMA") or target_schema).strip()  # Default to target schema
                source_table = (r.get("SOURCE_OBJ_NAME") or "").strip()
                target_obj_type = (r.get("TARGET_OBJ_TYPE") or "TABLE").strip().upper()
                
                # Validate required fields
                if not target_schema:
                    raise ValueError("TARGET_OBJ_SCHEMA is required but empty")
                if not target_table:
                    raise ValueError("TARGET_OBJ_NAME is required but empty")
                if not transform_query and not source_table:
                    raise ValueError("Either TRANSFORM_QUERY or SOURCE_OBJ_NAME is required")
                
                full_name = f"{CATALOG}.{target_schema}.{target_table}"
                
                print(f"  [{idx}/{len(rows)}] ▶ {full_name}")
                print(f"    Type: {target_obj_type} | Load: {load_type}")
                
                # Execute transformation or read source table
                try:
                    if transform_query:
                        print(f"    Transform: Custom SQL")
                        # Add catalog prefix if missing
                        if source_schema and f"{source_schema}." in transform_query and f"{CATALOG}.{source_schema}." not in transform_query:
                            transform_query = transform_query.replace(f"{source_schema}.", f"{CATALOG}.{source_schema}.")
                        
                        df = spark.sql(transform_query)
                    else:
                        print(f"    Source: {CATALOG}.{source_schema}.{source_table}")
                        df = spark.table(f"{CATALOG}.{source_schema}.{source_table}")
                except Exception as e:
                    raise RuntimeError(f"Failed to execute transformation: {type(e).__name__}: {str(e)}") from e
                
                # Validate DataFrame
                if df is None:
                    raise ValueError("Transformation returned None")
                
                # Add audit columns
                df = (
                    df
                    .withColumn("_etl_group_id", F.lit(GROUP_ID))
                    .withColumn("_etl_layer", F.lit(layer))
                    .withColumn("_etl_env", F.lit(ENV))
                    .withColumn("_etl_load_ts", F.current_timestamp())
                )
                
                # Write
                try:
                    count = write_table(df, CATALOG, target_schema, target_table, load_type, merge_keys)
                except Exception as e:
                    raise RuntimeError(f"Failed to write table: {type(e).__name__}: {str(e)}") from e
                
                if count == 0:
                    print(f"    ⚠ WARNING: 0 rows written")
                
                status = "SUCCESS"
                msg = f"{count:,} rows processed"
                print(f"    ✅ {msg}")
                success_count += 1
        
        except Exception as e:
            # FAIL LOUDLY - capture full error details
            status = "FAILED"
            error_type = type(e).__name__
            error_msg = str(e)
            msg = f"{error_type}: {error_msg[:300]}"
            
            print(f"    ❌ FAILED: {error_type}")
            print(f"    Error: {error_msg}")
            print(f"    Traceback:")
            traceback.print_exc()
            
            # Track failure
            failed_objects.append({
                "table": target_table or "unknown",
                "error": error_type,
                "message": error_msg
            })
        
        finally:
            t1 = datetime.now()
            duration = (t1 - t0).total_seconds()
            print(f"    ⏱ Duration: {duration:.1f}s\n")
            
            # Write audit log
            try:
                write_audit(GROUP_ID, target_table or "unknown", layer, status, msg, count, t0, t1)
            except Exception as audit_error:
                print(f"    ⚠ Audit write failed: {audit_error}")
    
    # Summary
    print(f"\n{'─'*60}")
    print(f"  Summary: {success_count}/{len(rows)} objects processed successfully")
    
    if failed_objects:
        print(f"  ❌ {len(failed_objects)} FAILURES:")
        for fail in failed_objects:
            print(f"    - {fail['table']}: {fail['error']}")
        print(f"{'─'*60}\n")
        
        # FAIL LOUDLY - raise exception to stop job
        error_summary = "; ".join([f"{f['table']}: {f['error']}" for f in failed_objects])
        raise RuntimeError(f"{len(failed_objects)} object(s) failed to process: {error_summary}")
    
    print(f"{'─'*60}\n")
    return True

In [0]:
# MAIN EXECUTION - FAIL LOUDLY ON ANY ERROR
start_time = datetime.now()

print(f"\n{'╔'+'═'*58+'╗'}")
print(f"║  ETL FRAMEWORK v2.0 - START                          ║")
print(f"║  {GROUP_ID:<50}  ║")
print(f"║  LAYER: {LAYER:<45}  ║")
print(f"║  TARGET: {(TARGET_TABLE or 'ALL'):<43}  ║")
print(f"{'╚'+'═'*58+'╝'}\n")

try:
    # Process based on layer
    # Note: process_layer raises exception on failure, so no need to check return value
    
    if LAYER == "L0":
        process_layer("data_flow_l0_detail", "L0")
    
    elif LAYER == "L1":
        process_layer("data_flow_pb_detail", "L1")
    
    elif LAYER == "L2":
        process_layer("data_flow_pb_detail", "L2")
    
    elif LAYER == "ALL":
        # Process all layers sequentially
        # If any layer fails, exception is raised immediately
        process_layer("data_flow_l0_detail", "L0")
        process_layer("data_flow_pb_detail", "L1")
        process_layer("data_flow_pb_detail", "L2")
    
    else:
        raise ValueError(f"Invalid LAYER: {LAYER}. Must be L0, L1, L2, or ALL.")
    
    # SUCCESS
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    
    print(f"\n{'╔'+'═'*58+'╗'}")
    print(f"║  ✅ ETL FRAMEWORK v2.0 - SUCCESS                     ║")
    print(f"║  Duration: {total_duration:.1f}s{' '*(43-len(str(int(total_duration))))}║")
    print(f"{'╚'+'═'*58+'╝'}\n")

except Exception as e:
    # FAIL LOUDLY - print full error and re-raise
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    error_type = type(e).__name__
    error_msg = str(e)
    
    print(f"\n{'╔'+'═'*58+'╗'}")
    print(f"║  ❌ ETL FRAMEWORK v2.0 - FAILED                      ║")
    print(f"║  Error: {error_type:<44}║")
    print(f"║  Duration: {total_duration:.1f}s{' '*(43-len(str(int(total_duration))))}║")
    print(f"{'╚'+'═'*58+'╝'}\n")
    
    print(f"\n{'='*60}")
    print(f"ERROR DETAILS:")
    print(f"  Type: {error_type}")
    print(f"  Message: {error_msg}")
    print(f"\nQuery audit log:")
    print(f"  SELECT * FROM {CATALOG}.admin.audit_log")
    print(f"  WHERE DATA_FLOW_GROUP_ID='{GROUP_ID}'")
    print(f"  ORDER BY LOAD_TS DESC")
    print(f"\nFull traceback:")
    print(f"{'='*60}")
    traceback.print_exc()
    
    # Re-raise to fail the job - NO SILENT ERRORS
    raise RuntimeError(f"ETL Framework failed: {error_type}: {error_msg}") from e